In [ ]:
# 必要なものを用意
## ライブラリ
!pip install scikit-learn
## データセット
from datasets import load_dataset
clinc = load_dataset("clinc_oos", "plus")
## 評価関数
from sklearn.metrics import accuracy_score
## パイプライン
from transformers import pipeline
bert_ckpt = "transformersbook/bert-base-uncased-finetuned-clinc"
pipe = pipeline("text-classification", model=bert_ckpt)
## 意図
intents = clinc["test"].features["intent"]
## ベンチマーク用クラス
import torch
from pathlib import Path
from time import perf_counter
import numpy as np
class PerformanceBenchmark:
    def __init__(self, pipeline, dataset, optim_type="BERT baseline"):
        self.pipeline = pipeline
        self.dataset = dataset
        self.optim_type = optim_type

    def compute_accuracy(self):
        preds, labels = [], []
        for example in self.dataset:
            pred = self.pipeline(example["text"])[0]["label"]
            label = example["intent"]
            preds.append(intents.str2int(pred))
            labels.append(label)
        accuracy = accuracy_score(labels, preds)
        print(f"Accuracy on test set - {accuracy:.3f}")
        return {"accuracy": accuracy}

    def compute_size(self):
        temp_path = Path("model.pt")
        torch.save(self.pipeline.model.state_dict(), temp_path)
        size_mb = temp_path.stat().st_size / (1024 * 1024)
        print(f"Model size - {size_mb:.2f} MB")
        temp_path.unlink()
        return {"size_mb": size_mb}

    def time_pipeline(self, query="What is the pin number for my account?"):
        latencies = []
        for _ in range(10):
            _ = self.pipeline(query)
        for _ in range(100):
            start_time = perf_counter()
            _ = self.pipeline(query)
            latency = perf_counter() - start_time
            latencies.append(latency)
        time_avg_ms = 1000 * np.mean(latencies)
        time_std_ms = 1000 * np.std(latencies)
        print(f"Average latency (ms) - {time_avg_ms:.2f} +\- {time_std_ms:.2f}")
        return {"time_avg_ms": time_avg_ms, "time_std_ms": time_std_ms}

    def run_benchmark(self):
        metrics = {}
        metrics[self.optim_type] = self.compute_size()
        metrics[self.optim_type].update(self.time_pipeline())
        metrics[self.optim_type].update(self.compute_accuracy())
        return metrics
## ベースライン
pb = PerformanceBenchmark(pipe, clinc["test"])
perf_metrics = pb.run_benchmark()
## 生徒モデル
finetuned_ckpt = "kirapika2/distilbert-base-uncased-finetuned-clinc"
pipe = pipeline("text-classification", model=finetuned_ckpt)
optim_type = "DistilBERT"
pb = PerformanceBenchmark(pipe, clinc["test"], optim_type=optim_type)
perf_metrics.update(pb.run_benchmark())
## 蒸留モデル
distilled_ckpt = "kirapika2/distilbert-base-uncased-distilled-clinc"
pipe = pipeline("text-classification", model=distilled_ckpt)
optim_type = "Distillation"
pb = PerformanceBenchmark(pipe, clinc["test"], optim_type=optim_type)
perf_metrics.update(pb.run_benchmark())
## 量子化モデル
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.quantization import quantize_dynamic
tokenizer = AutoTokenizer.from_pretrained(distilled_ckpt)
model = (AutoModelForSequenceClassification
         .from_pretrained(distilled_ckpt).to("cpu"))
model_qunantized = quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)
pipe = pipeline("text-classification", model=model_qunantized,
                tokenizer=tokenizer, device=-1)
optim_type = "Distillation + Quantization"
pb = PerformanceBenchmark(pipe, clinc["test"], optim_type=optim_type)
perf_metrics.update(pb.run_benchmark())
## プロット
import pandas as pd
import matplotlib.pyplot as plt
def plot_metrics(perf_metrics, current_optim_type):
    df = pd.DataFrame.from_dict(perf_metrics, orient='index')

    for idx in df.index:
        df_opt = df.loc[idx]
        # 現在の最適化タイプを中抜きの円で強調
        if idx == current_optim_type:
            plt.scatter(df_opt["time_avg_ms"], df_opt["accuracy"] * 100,
                        s=df_opt["size_mb"], label=idx,
                        facecolors='none', edgecolors='C0', linewidths=2)
        else:
            plt.scatter(df_opt["time_avg_ms"], df_opt["accuracy"] * 100,
                        s=df_opt["size_mb"], label=idx, alpha=0.5)

    legend = plt.legend(bbox_to_anchor=(1,1))
    for handle in getattr(legend, "legend_handles", getattr(legend, "legendHandles", [])):
        if hasattr(handle, "set_sizes"):
            handle.set_sizes([20])

    plt.ylim(80,90)
    # 最も遅いモデルを使い、x軸のレンジを定義
    xlim = int(perf_metrics["BERT baseline"]["time_avg_ms"] + 3)
    plt.xlim(1,xlim)
    plt.ylabel("Accuracy (%)")
    plt.xlabel("Average latency (ms)")
    plt.show()
## クエリの前処理
def tokenize_text(batch):
    return tokenizer(batch["text"], truncation=True, return_token_type_ids=False)
clinc_enc = clinc.map(tokenize_text, batched=True, remove_columns=["text"])
clinc_enc = clinc_enc.rename_column("intent", "labels")

In [ ]:
# OpenMP設定
import os
from psutil import cpu_count

os.environ["OMP_NUM_THREADS"] = f"{cpu_count()}"
os.environ["OMP_WAIT_POLICY"] = "ACTIVE"

In [ ]:
# ONNX 変換に必要なライブラリ
%pip install -q onnx onnxruntime-gpu

In [ ]:
# モデルの ONNX 形式への変換
model_ckpt = "kirapika2/distilbert-base-uncased-distilled-clinc"
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt)
model.eval()

onnx_dir = Path("onnx")
onnx_dir.mkdir(parents=True, exist_ok=True)
onnx_model_path = onnx_dir / "model.onnx"

sample_inputs = tokenizer(
    "What is the pin number for my account?",
    return_tensors="pt",
)

with torch.no_grad():
    torch.onnx.export(
        model,
        args=(sample_inputs["input_ids"], sample_inputs["attention_mask"]),
        f=onnx_model_path.as_posix(),
        input_names=["input_ids", "attention_mask"],
        output_names=["logits"],
        dynamic_axes={
            "input_ids": {0: "batch_size", 1: "sequence_length"},
            "attention_mask": {0: "batch_size", 1: "sequence_length"},
            "logits": {0: "batch_size"},
        },
        opset_version=14,
        dynamo=False,
    )

print(f"ONNX model exported to: {onnx_model_path}")

In [ ]:
# ONNX モデルの推論インスタンスの作成
from onnxruntime import (GraphOptimizationLevel, InferenceSession, SessionOptions)

def create_model_for_provider(onnx_model_path, provider="CUDAExecutionProvider"):
    options = SessionOptions()
    options.intra_op_num_threads = 1
    options.graph_optimization_level = GraphOptimizationLevel.ORT_ENABLE_ALL
    session = InferenceSession(onnx_model_path.as_posix(), options, providers=[provider])
    return session

onnx_model = create_model_for_provider(onnx_model_path)


In [ ]:
# ONNX モデルの出力を取得
inputs = clinc_enc["test"][:1]
del inputs["labels"]
logits_onnx = onnx_model.run(None, inputs)[0]
logits_onnx.shape # (1, 151)

In [ ]:
# 予測ラベルと正解ラベルの比較
print(np.argmax(logits_onnx)) # 61
print(clinc_enc["test"][0]["labels"]) # 61

In [ ]:
# ONNX モデルのパイプラインを作成
from scipy.special import softmax

class OnnxPipeline:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    def __call__(self, query):
        model_inputs = self.tokenizer(query, return_tensors="pt", return_token_type_ids=False)
        inputs_onnx = {k: v.cpu().detach().numpy()
                       for k, v in model_inputs.items()}
        logits = self.model.run(None, inputs_onnx)[0][0, :]
        probs = softmax(logits)
        pred_idx = np.argmax(probs).item()
        return [{"label": intents.int2str(pred_idx), "score": probs[pred_idx]}]

In [ ]:
# 単純なクエリで確認
query = """Hey, I'd like to rent a vehicle from Nov 1st to Nov 15th in
Paris and I need a 15 passenger van"""
pipe = OnnxPipeline(onnx_model, tokenizer)
pipe(query)

In [ ]:
# ONNX モデルの推論ベンチマーク
class OnnxPerformanceBenchmark(PerformanceBenchmark):
    def __init__(self, *args, model_path, **kwargs):
        super().__init__(*args, **kwargs)
        self.model_path = model_path

    def compute_size(self):
        size_mb = Path(self.model_path).stat().st_size / (1024 * 1024)
        print(f"Model size (MB) - {size_mb:.2f}")
        return {"size_mb": size_mb}

In [ ]:
# ベンチマーク実行
optim_type = "Distillation + ORT"
pb = OnnxPerformanceBenchmark(pipe, clinc["test"], optim_type, model_path=onnx_model_path)
perf_metrics.update(pb.run_benchmark())

In [ ]:
# ベンチマーク結果を描画
plot_metrics(perf_metrics, optim_type)

In [ ]:
# ONNX モデルの量子化
from onnxruntime.quantization import quantize_dynamic, QuantType

model_input = onnx_model_path.as_posix()
model_output = onnx_dir / "model.quant.onnx"
quantize_dynamic(model_input, model_output.as_posix(), weight_type=QuantType.QInt8)

In [ ]:
# 量子化 ONNX モデルのベンチマーク
onnx_quantized_model = create_model_for_provider(model_output)
pipe = OnnxPipeline(onnx_quantized_model, tokenizer)
optim_type = "Distillation + ORT (quantized)"
pb = OnnxPerformanceBenchmark(pipe, clinc["test"], optim_type, model_path=model_output)
perf_metrics.update(pb.run_benchmark())

In [ ]:
print(onnx_quantized_model.get_providers())

In [ ]:
# 結果表示
plot_metrics(perf_metrics, optim_type)